# NOTA: 5.0

Buen trabajo !!!

# Realización del trabajo 1.

__Nombres:__

- Julian Esteban Ariza Caro CC 1115724987
- Mauricio David Timana Getial CC 1087408976

In [ ]:
 # Importamos las librerías necesarias para manipulación, análisis y visualización de datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import unicodedata

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Nombre del archivo que buscan
nombre_archivo = "/content/drive/MyDrive/PLE/DATOS/MEN_MATRICULA_ESTADISTICA_ES.csv"
ruta_encontrada = None

# 3. Buscar el archivo automáticamente en cualquier carpeta de Mi Unidad
for ruta_actual, _, archivos in os.walk("/content/drive/MyDrive"):
    if nombre_archivo in archivos:
        ruta_encontrada = os.path.join(ruta_actual, nombre_archivo)
        break

# 4. Cargar la base de datos
if ruta_encontrada:
    print(f"Archivo cargado desde: {ruta_encontrada}")
    estudiantes = pd.read_csv(ruta_encontrada, thousands='.', low_memory=False)
else:
    print("No se encontró el archivo en Google Drive. Verifica que esté subido o compartido.")

## Limpieza de la base de datos.

In [ ]:
# Inspeccion inicial: dimensiones, primeras filas y estructura del data frame

print(estudiantes.shape)
estudiantes.head()

In [ ]:
# Muestra el resumen del tipo de datos por columna y conteo de valores no nulos
print(estudiantes.info())

In [ ]:


# 1. Buscamos '0' y 'Sin clasificar' en TODAS las columnas del DataFrame
res = []
for col in estudiantes.columns:
    col_str = estudiantes[col].astype(str).str.strip().str.upper()
    c_zero = (col_str == "0") | (col_str == "0.0")
    c_sin = col_str.str.contains("SIN CLASIFICAR")
    c_total = (c_zero | c_sin).sum()
    if c_total > 0:
        res.append(
            {
                "Columna": col,
                "Ceros": c_zero.sum(),
                "Sin Clasificar": c_sin.sum(),
                "Total Faltantes": c_total,
                "% Faltante": round(100 * c_total / estudiantes.shape[0], 2),
            }
        )

# 2. Convertimos el resultado a DataFrame para visualizarlo claramente
estudiantes_faltantes = pd.DataFrame(res)
print(estudiantes_faltantes)

In [ ]:


# 1. Eliminamos las filas exactamente duplicadas y creamos una copia limpia
estudiantes_limpio = estudiantes.drop_duplicates().copy()

# 2. Imprimimos el resultado del procedimiento
print(f"Filas en base original : {estudiantes.shape[0]}")
print(f"Filas en base depurada : {estudiantes_limpio.shape[0]}")
print(f"Total filas duplicadas eliminadas: {estudiantes.shape[0] - estudiantes_limpio.shape[0]}")

# 3. Comprobamos que no queden duplicados
print(f"\n¿Quedan filas duplicadas en estudiantes_limpio?: {estudiantes_limpio.duplicated().any()}")

In [ ]:


# 1. Definimos las columnas a estandarizar
cols_faltantes = [
    "Id_Area",
    "Id_Nucleo",
    "Núcleo Básico del Conocimiento (NBC)",
]

# 2. Reemplazamos 'Sin clasificar', '0' y '0.0' por np.nan
for col in cols_faltantes:
    estudiantes_limpio[col] = (
        estudiantes_limpio[col]
        .astype(str)
        .str.strip()
        .replace(["Sin clasificar", "0", "0.0"], np.nan)
    )

# 3. Verificamos el nuevo conteo de valores nulos reconocidos por Pandas
print("=== CONTEO DE NULOS EN df_limpio ===")
print(estudiantes_limpio[cols_faltantes].isna().sum())

# 4. Verificamos el porcentaje de nulos por columna
print("\n=== PORCENTAJE DE NULOS POR COLUMNA (%) ===")
print(100 * round(estudiantes_limpio[cols_faltantes].isna().sum() / estudiantes_limpio.shape[0], 4))

In [ ]:


# 1. Mapeo oficial a partir del Código DANE del Departamento
diccionario_dane_deptos = {
    5: "ANTIOQUIA",
    8: "ATLÁNTICO",
    11: "BOGOTÁ, D.C.",
    13: "BOLÍVAR",
    15: "BOYACÁ",
    17: "CALDAS",
    18: "CAQUETÁ",
    19: "CAUCA",
    20: "CESAR",
    23: "CÓRDOBA",
    25: "CUNDINAMARCA",
    27: "CHOCÓ",
    41: "HUILA",
    44: "LA GUAJIRA",
    47: "MAGDALENA",
    50: "META",
    52: "NARIÑO",
    54: "NORTE DE SANTANDER",
    63: "QUINDÍO",
    66: "RISARALDA",
    68: "SANTANDER",
    70: "SUCRE",
    73: "TOLIMA",
    76: "VALLE DEL CAUCA",
    81: "ARAUCA",
    85: "CASANARE",
    86: "PUTUMAYO",
    88: "ARCHIPIÉLAGO DE SAN ANDRÉS, PROVIDENCIA Y SANTA CATALINA",
    91: "AMAZONAS",
    94: "GUAINÍA",
    95: "GUAVIARE",
    97: "VAUPÉS",
    99: "VICHADA",
}

# 2. Normalizamos los departamentos en IES y Programa
estudiantes_limpio["Departamento de domicilio de la IES"] = estudiantes_limpio[
    "Código del departamento(IES)"
].map(diccionario_dane_deptos)
estudiantes_limpio["Departamento de oferta del programa"] = estudiantes_limpio[
    "Código del Departamento(Programa)"
].map(diccionario_dane_deptos)

# 3. Comprobamos que no hayan quedado valores nulos
print(
    "Nulos en Departamento IES:",
    estudiantes_limpio["Departamento de domicilio de la IES"].isna().sum(),
)
print(
    "Nulos en Departamento Programa:",
    estudiantes_limpio["Departamento de oferta del programa"].isna().sum(),
)

# 4. Verificamos la cantidad de departamentos únicos resultantes
print(
    "\nTotal departamentos únicos normalizados:",
    estudiantes_limpio["Departamento de oferta del programa"].nunique(),
)

In [ ]:

# 1. Estandarizamos los nombres de las columnas a minúsculas, sin tildes ni espacios
estudiantes_limpio.columns = (
    estudiantes_limpio.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "_")
    .str.replace(")", "")
    .str.replace("á", "a")
    .str.replace("é", "e")
    .str.replace("í", "i")
    .str.replace("ó", "o")
    .str.replace("ú", "u")
    .str.replace("ñ", "n")
)

# 2. Imprimimos la lista limpia de columnas
print(estudiantes_limpio.columns.tolist())

In [ ]:


print(f"Dimensiones finales (filas, columnas): {estudiantes_limpio.shape}")
print(
    f"Filas eliminadas respecto al original: {estudiantes.shape[0] - estudiantes_limpio.shape[0]}"
)
print(f"Total registros duplicados restantes: {estudiantes_limpio.duplicated().sum()}")

print(estudiantes_limpio.isna().sum()[estudiantes_limpio.isna().sum() > 0])

print(estudiantes_limpio.head(3))

In [ ]:


# 1. Eliminamos los duplicados que aparecieron tras unificar los nombres de departamento
estudiantes_limpio = estudiantes_limpio.drop_duplicates().copy()

# 2. Verificación final de dimensiones y duplicados
print(f"Dimensiones finales del dataset: {estudiantes_limpio.shape}")
print(f"Total registros duplicados restantes: {estudiantes_limpio.duplicated().sum()}")

In [ ]:
estudiantes_limpio.head()

In [ ]:
# 1. Copia explícita para evitar warnings de rebanado
estudiantes_limpio = estudiantes_limpio.copy()


# 2. Función para remover tildes, signos de puntuación y espacios extras
def normalizar_texto(texto):
    if pd.isna(texto) or not isinstance(texto, str):
        return texto
    # Descomponer caracteres con tildes/acentos
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    # Remover comas, puntos y guiones
    texto = (
        texto.replace(",", "").replace(".", "").replace("-", " ").strip().upper()
    )
    return " ".join(texto.split())


# 3. Selección de columnas de texto
cols_str = estudiantes_limpio.select_dtypes(include=["object"]).columns

# 4. Aplicación de la limpieza profunda
for col in cols_str:
    estudiantes_limpio[col] = estudiantes_limpio[col].apply(normalizar_texto)

# 5. Mapeo de valores nulos a NaN
estudiantes_limpio.loc[:, cols_str] = estudiantes_limpio[cols_str].replace(
    ["NAN", "NONE", "N/A", "NA", "SIN CLASIFICAR", "0", ""], np.nan
)

# 6. Depuración final de duplicados
estudiantes_limpio = estudiantes_limpio.drop_duplicates().reset_index(
    drop=True
)

In [ ]:
estudiantes_limpio

In [ ]:
# Definir la lista exacta de variables a revisar
cols_revisar = [

    "id_area",
    "id_nucleo",
    "nucleo_basico_del_conocimiento__nbc"
]

# Vista previa de los primeros 10 registros
print("=== VISTA PREVIA DE LAS VARIABLES SELECCIONADAS ===")
print(estudiantes_limpio[cols_revisar].head(10))

print("\n" + "=" * 80 + "\n")

# Detalle de valores únicos e información general por variable
print("=== VALORES ÚNICOS Y TIPOS DE DATOS ===")
for col in cols_revisar:
    if col in estudiantes_limpio.columns:
        n_nulos = estudiantes_limpio[col].isna().sum()
        n_unicos = estudiantes_limpio[col].nunique()
        tipo = estudiantes_limpio[col].dtype

        print(f" Variable: '{col}' | Tipo: {tipo}")
        print(f" Registros no nulos: {estudiantes_limpio[col].count():,} | Nulos: {n_nulos:,} | Valores únicos: {n_unicos:,}"
        )

        # Si la columna es categórica/texto, mostramos una muestra de sus valores
        if tipo == "object":
            muestra = sorted(
                [str(v) for v in estudiantes_limpio[col].dropna().unique()]
            )
            print(f" Ejemplos de valores: {muestra[:5]}")

        print("-" * 60)

In [ ]:


# Copia de seguridad del estado actual
estudiantes_limpio = estudiantes_limpio.copy()

# Guardar recuento inicial
filas_antes = len(estudiantes_limpio)

# 1. Regra de Julián: Eliminar filas donde 'id_nucleo' Y 'nucleo_basico_del_conocimiento__nbc' son NaN al mismo tiempo
estudiantes_limpio = estudiantes_limpio.dropna(
    subset=["id_nucleo", "nucleo_basico_del_conocimiento__nbc"], how="all"
).reset_index(drop=True)

filas_despues = len(estudiantes_limpio)
print(f"Filas eliminadas (sin ID ni NBC): {filas_antes - filas_despues:,}")
print(f"Total filas restantes: {filas_despues:,}")

In [ ]:
# 2. Construir diccionarios de mapeo basados en el código SNIES del programa
mapa_snies = (
    estudiantes_limpio.dropna(
        subset=["id_area", "id_nucleo", "nucleo_basico_del_conocimiento__nbc"]
    )
    .drop_duplicates(subset=["codigo_snies_delprograma"])[
        [
            "codigo_snies_delprograma",
            "id_area",
            "id_nucleo",
            "nucleo_basico_del_conocimiento__nbc",
        ]
    ]
    .set_index("codigo_snies_delprograma")
)

# 3. Imputar/recuperar valores nulos faltantes en id_area, id_nucleo y NBC
cols_recuperar = [
    "id_area",
    "id_nucleo",
    "nucleo_basico_del_conocimiento__nbc",
]

for col in cols_recuperar:
    estudiantes_limpio[col] = estudiantes_limpio[col].fillna(
        estudiantes_limpio["codigo_snies_delprograma"].map(mapa_snies[col])
    )

# 4. Verificación de nulos resultantes tras la recuperación
print("\n=== REGISTROS NULOS TRAS LA RECUPERACIÓN ===")
for col in cols_recuperar:
    print(f"Nulos en '{col}': {estudiantes_limpio[col].isna().sum():,}")

# 5. Guardar dataset actualizado
estudiantes_limpio.to_csv(
    "df_limpio_ple.csv", index=False, encoding="utf-8-sig"
)

In [ ]:
estudiantes_limpio.info()

In [ ]:
estudiantes_limpio

In [ ]:
# Mostrar todos los índices donde NBC es nulo
nulos_nbc = estudiantes_limpio[
    estudiantes_limpio["nucleo_basico_del_conocimiento__nbc"].isna()
]["nucleo_basico_del_conocimiento__nbc"]

print(
    f"Total de registros nulos en NBC: {len(nulos_nbc)}\n"
)
print(nulos_nbc.to_string())

In [ ]:
# 1. Crear mapa basado en el nombre del programa académico
mapa_por_programa = (
    estudiantes_limpio.dropna(subset=["nucleo_basico_del_conocimiento__nbc"])
    .drop_duplicates(subset=["programa_academico"])[
        ["programa_academico", "nucleo_basico_del_conocimiento__nbc"]
    ]
    .set_index("programa_academico")["nucleo_basico_del_conocimiento__nbc"]
)

# 2. Imputar los nulos restantes en la columna NBC
estudiantes_limpio["nucleo_basico_del_conocimiento__nbc"] = (
    estudiantes_limpio["nucleo_basico_del_conocimiento__nbc"].fillna(
        estudiantes_limpio["programa_academico"].map(mapa_por_programa)
    )
)

# 3. Guardar cambios en el archivo CSV
estudiantes_limpio.to_csv(
    "df_limpio_ple.csv", index=False, encoding="utf-8-sig"
)

# 4. Verificar nulos restantes tras la imputación
nulos_finales = estudiantes_limpio[
    "nucleo_basico_del_conocimiento__nbc"
].isna().sum()
print(f"Nulos restantes en NBC tras la imputación: {nulos_finales}")

In [ ]:
# Eliminar las 16 filas sin clasificación
estudiantes_limpio = estudiantes_limpio.dropna(
    subset=["nucleo_basico_del_conocimiento__nbc"]
).reset_index(drop=True)

# Guardar base final
estudiantes_limpio.to_csv(
    "df_limpio_ple.csv", index=False, encoding="utf-8-sig"
)

print(f"Total de filas finales en la base: {len(estudiantes_limpio):,}")
print(
    f"Total nulos en todo el dataset: {estudiantes_limpio.isna().sum().sum()}"
)

In [ ]:
estudiantes_limpio.info()

In [ ]:
estudiantes_limpio['id_genero'].value_counts()

In [ ]:
estudiantes_limpio.info()

In [ ]:
estudiantes_limpio['id_area'].value_counts()

In [ ]:
estudiantes_limpio['id_metodologia'].value_counts()

In [ ]:

estudiantes_limpio = estudiantes_limpio.dropna()

In [ ]:
estudiantes_limpio.head(1000)

In [ ]:
estudiantes_limpio

In [ ]:
estudiantes_limpio['municipio_de_oferta_del_programa'].value_counts()

In [ ]:
estudiantes_limpio.columns


In [ ]:
estudiantes_limpio['total_matriculados'].value_counts()

In [ ]:
estudiantes_limpio['codigo_de_la_institucion'].value_counts()

In [ ]:
estudiantes_limpio['institucion_de_educacion_superior__ies'].value_counts()

In [ ]:
estudiantes_limpio['ies_padre'].value_counts()

In [ ]:
# 1. Convertir la columna a mayúsculas y quitar espacios extra
estudiantes_limpio['institucion_de_educacion_superior__ies'] = (
    estudiantes_limpio['institucion_de_educacion_superior__ies']
    .str.upper()
    .str.strip()
)

# 2. Verificar el nuevo número de nombres únicos
total_nombres_mayus = estudiantes_limpio['institucion_de_educacion_superior__ies'].nunique()
print(f"Total de nombres únicos tras convertir a mayúsculas: {total_nombres_mayus}")

In [ ]:
# Limpieza de texto en nombres de IES
estudiantes_limpio['institucion_de_educacion_superior__ies'] = (
    estudiantes_limpio['institucion_de_educacion_superior__ies']
    .astype(str)
    .str.replace('¿', 'Ñ')
    .str.upper()
    .str.strip()
)

# Homologar nombres asignando el registro más reciente a cada código
df_temp = estudiantes_limpio.sort_values(['ano', 'semestre'], ascending=False)
mapa_nombres = df_temp.groupby('codigo_de_la_institucion')['institucion_de_educacion_superior__ies'].first()

estudiantes_limpio['institucion_de_educacion_superior__ies'] = (
    estudiantes_limpio['codigo_de_la_institucion'].map(mapa_nombres)
)

# Validación de conteos
print("Códigos de IES:", estudiantes_limpio['codigo_de_la_institucion'].nunique())
print("Nombres de IES unificados:", estudiantes_limpio['institucion_de_educacion_superior__ies'].nunique())

In [ ]:
estudiantes_limpio['institucion_de_educacion_superior__ies'].value_counts()

In [ ]:
# 1. Verificar que los códigos únicos se mantienen
print("Total códigos de IES:", estudiantes_limpio['codigo_de_la_institucion'].nunique())

# 2. Ver las universidades que agrupan varias sedes/códigos
codigos_por_nombre = estudiantes_limpio.groupby('institucion_de_educacion_superior__ies')['codigo_de_la_institucion'].nunique()
print("\nUniversidades con más de una sede/código registrado:")
print(codigos_por_nombre[codigos_por_nombre > 1].sort_values(ascending=False))

In [ ]:
estudiantes_limpio['nucleo_basico_del_conocimiento__nbc'].value_counts()

In [ ]:
estudiantes_limpio['codigo_snies_delprograma'].value_counts()

In [ ]:
estudiantes_limpio['departamento_de_oferta_del_programa'].value_counts()

In [ ]:
estudiantes_limpio['programa_academico'].value_counts()

In [ ]:
# 1. Diccionario con los patrones de reemplazo más comunes en los nombres de programas
reemplazos = {
    '¿N': 'ÓN',      # INFORMACI¿N -> INFORMACIÓN, ADMINISTRACI¿N -> ADMINISTRACIÓN
    'G¿A': 'GÍA',    # TECNOLOG¿A -> TECNOLOGÍA, PSICOLOG¿A -> PSICOLOGÍA
    'R¿A': 'RÍA',    # INGENIER¿A -> INGENIERÍA, CONTADUR¿A -> CONTADURÍA
    'N¿L': 'NÁL',    # AN¿LISIS -> ANÁLISIS
    '¿A': 'ÍA',      # Otros casos terminados en ÍA
    '¿O': 'Ó',
    '¿E': 'É',
    '¿I': 'Í',
    '¿U': 'Ú'
}

# 2. Aplicar la corrección a la columna 'programa_academico'
for viejo, nuevo in reemplazos.items():
    estudiantes_limpio['programa_academico'] = estudiantes_limpio['programa_academico'].str.replace(viejo, nuevo, regex=False)

# --- Volver a ejecutar el punto (m) con el nombre corregido ---
prog_ano = estudiantes_limpio[estudiantes_limpio['ano'].isin([2015, 2021])].groupby(['programa_academico', 'ano'])['total_matriculados'].sum().unstack(fill_value=0)
crecimiento = prog_ano[2021] - prog_ano[2015]

print("Programa académico con mayor crecimiento absoluto (2015-2021):")
print(f"{crecimiento.idxmax()}: +{crecimiento.max():,.0f} matriculados")

In [ ]:
estudiantes_limpio.info()

In [ ]:
estudiantes_limpio.columns

## Solución de las preguntas.

__(a).__ ¿Cuántas instituciones de educación superior (IES) distintas hay en la
BD y en qué año se reportó el mayor número de IES distintas?

In [ ]:
# Utlizaremos el codigo de la institucion como referencia
total_ies = estudiantes_limpio['codigo_de_la_institucion'].nunique()
print(f"Total de IES distintas en la base de datos: {total_ies}")

# Conteo de IES distintas por año
ies_por_ano = estudiantes_limpio.groupby('ano')['codigo_de_la_institucion'].nunique()
print("\nIES distintas por año:")
print(ies_por_ano)

# Año con el mayor número de IES
ano_max = ies_por_ano.idxmax()
cant_max = ies_por_ano.max()

print(f"\nEl año con mayor número de IES distintas fue {ano_max} con {cant_max} instituciones.")

__(b).__ ¿Cuál fue el total de matriculados en cada año entre 2015 y 2021 y en qué año se presentó el mayor y el menor crecimiento porcentual respecto al
año anterior?

In [ ]:
# 1. Total de matriculados por año (2015 - 2021)
matriculados_por_año = estudiantes_limpio.groupby('ano')['total_matriculados'].sum().reset_index()

# 2. Crecimiento porcentual respecto al año anterior
matriculados_por_año['crecimiento_porcentaje'] = matriculados_por_año['total_matriculados'].pct_change() * 100

print("Total matriculados y crecimiento porcentual por año:")
print(matriculados_por_año)

# 3. Identificar el año con mayor y menor crecimiento porcentual (excluyendo 2015 por ser el año inicial)
datos_crecimiento = matriculados_por_año.dropna(subset=['crecimiento_porcentaje'])

max_crec = datos_crecimiento.loc[datos_crecimiento['crecimiento_porcentaje'].idxmax()]
min_crec = datos_crecimiento.loc[datos_crecimiento['crecimiento_porcentaje'].idxmin()]

print(f"\nMayor crecimiento porcentual: Año {int(max_crec['ano'])} con {max_crec['crecimiento_porcentaje']:.2f}%")
print(f"Menor crecimiento porcentual: Año {int(min_crec['ano'])} con {min_crec['crecimiento_porcentaje']:.2f}%")

In [ ]:
# Gráfica de línea para el total de matriculados por año
plt.figure(figsize=(10, 5))
plt.plot(matriculados_por_año['ano'], matriculados_por_año['total_matriculados'], marker='o', color='#1f77b4', linewidth=2)

plt.title('Evolución del Total de Matriculados en Educación Superior (2015 - 2021)', fontsize=12, pad=15)
plt.xlabel('Año', fontsize=10)
plt.ylabel('Total de Matriculados', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)

# Etiquetas con los valores sobre cada punto de la gráfica
for idx, fila in matriculados_por_año.iterrows():
    plt.annotate(
        f"{int(fila['total_matriculados']):,}".replace(",", "."),
        (fila['ano'], fila['total_matriculados']),
        textcoords="offset points",
        xytext=(0, 8),
        ha='center',
        fontsize=9
    )

plt.tight_layout()
plt.show()

Como se puede ver en la grafica anterior y en la tabla del anterior codigo el año que presento un mayor crecimiento porcentual es 2018 con 3.37% y el que presentó menor crecimiento porcentual es 2020 con -2.20% con respecto al año anterior.

__(c).__ ¿Cuál es la IES con mayor número de matriculados en 2019?

In [ ]:
estudiantes_2019 = estudiantes_limpio[estudiantes_limpio['ano'] == 2019]
IES_2019 = estudiantes_2019.groupby('institucion_de_educacion_superior__ies')['total_matriculados'].sum().sort_values(ascending=False)

IES_2019.head(1)

El SENA reporta 431109 matriculados en el año 2019, con esta cifra es la IES con mayor número de matriculados ese año.

__(d).__ ¿Cuántos programas académicos distintos (según código SNIES) se re
portaron en cada año y en cuál año hubo más?

In [ ]:
programas_por_año = estudiantes_limpio.groupby('ano')['codigo_snies_delprograma'].nunique()

print(programas_por_año)

# Año con mayor cantidad de programas
ano_max = programas_por_año.idxmax()
cant_max = programas_por_año.max()

print(f"\nEl año con más programas reportados fue {ano_max} con {cant_max} programas distintos.")

Teniendo en cuenta el código SNIES del programa académico, el año con más programas reportados fue el año 2021 con 11807 programas distintos.

__(e).__ ¿Cuál es la proporción de mujeres y de hombres matriculados en cada
año y en qué año fue mayor la diferencia entre ambos?

In [ ]:
# Suma total por año y género
tabla = estudiantes_limpio.groupby(['ano', 'id_genero'])['total_matriculados'].sum().unstack()
tabla.columns = ['Hombres', 'Mujeres']

# Proporción (entre 0 y 1)
prop = tabla.div(tabla.sum(axis=1), axis=0)

# Diferencia entre ambos
prop['Diferencia'] = (prop['Mujeres'] - prop['Hombres']).abs()

print("Proporción de hombres y mujeres por año:")
print(prop[['Hombres', 'Mujeres']])

print("\nAño con mayor diferencia:", prop['Diferencia'].idxmax())

In [ ]:
# Agrupar por año y género
genero_año = estudiantes_limpio.groupby(['ano', 'id_genero'])['total_matriculados'].sum().unstack()
genero_año.columns = ['Hombres', 'Mujeres']

# Crear grilla de gráficas circulares
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for i, (ano, fila) in enumerate(genero_año.iterrows()):
    axes[i].pie(fila, labels=fila.index, autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'], startangle=90)
    axes[i].set_title(f'Año {ano}')

fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

En la tabla se puede ver las proporciones de mujeres y de hombres matriculados en cada año, el año que fue mayor la diferencia en ambos fue el año 2021 teniendo en cuenta las graficas circulares con  53.3% de mujeres.

__(f).__ En 2021, ¿cuál es el núcleo básico del conocimiento con mayor proporción de mujeres matriculadas y cuál con menor proporción?

In [ ]:
# Filtrar 2021
estudiantes_2021 = estudiantes_limpio[estudiantes_limpio['ano'] == 2021]

# Agrupar por NBC y género
nbc = estudiantes_2021.groupby(['nucleo_basico_del_conocimiento__nbc', 'id_genero'])['total_matriculados'].sum().unstack(fill_value=0)
nbc.columns = ['Hombres', 'Mujeres']

# Proporción de mujeres sobre el total
prop_mujeres = nbc['Mujeres'] / (nbc['Hombres'] + nbc['Mujeres'])

# Mayor y menor proporción de mujeres
print("Mayor proporción de mujeres:", prop_mujeres.idxmax(), prop_mujeres.max())
print("Menor proporción de mujeres:", prop_mujeres.idxmin(), prop_mujeres.min())

In [ ]:

# Agrupar por NBC y género
nbc = estudiantes_2021.groupby(['nucleo_basico_del_conocimiento__nbc', 'id_genero'])['total_matriculados'].sum().unstack(fill_value=0)
nbc.columns = ['Hombres', 'Mujeres']

# Calcular proporción de mujeres
nbc['prop_mujeres'] = nbc['Mujeres'] / (nbc['Hombres'] + nbc['Mujeres'])

# Graficar
plt.figure(figsize=(10, 12))
nbc['prop_mujeres'].sort_values().plot(kind='barh', color='skyblue')

plt.title('Proporción de mujeres por NBC (2021)')
plt.xlabel('Proporción')
plt.ylabel('NBC')
plt.axvline(0.5, color='red', linestyle='--')
plt.tight_layout()
plt.show()

Como se puede ver en la grafica el núcleo básico del conocimiento con mayor proporción de mujeres matriculadas es psicología y el menor proporción es ingeniería mecánica  y afines.

__(g).__ ¿Cómo cambió la participación del sector oficial dentro del total de matriculados entre 2015 y 2021?

In [ ]:
tabla_sector = estudiantes_limpio.groupby(['ano', 'id_sector'])['total_matriculados'].sum().unstack()
tabla_sector.columns = ['Oficial', 'Privado']

# Calcular porcentaje de participación
part_sector = tabla_sector.div(tabla_sector.sum(axis=1), axis=0) * 100

# Participación en 2015 y 2021
p_2015 = part_sector.loc[2015, 'Oficial']
p_2021 = part_sector.loc[2021, 'Oficial']
cambio = p_2021 - p_2015

print(f"Participación Oficial 2015: {p_2015:.2f}%")
print(f"Participación Oficial 2021: {p_2021:.2f}%")
print(f"{cambio:+.2f} puntos porcentuales")

En el año 2015 el sector oficial del total de matriculados fue 50.92% y en el año 2021 fue 53.89%, esto quiere decir que cambio 2.9% el sector oficial en esos años.

__(h).__ Compare 2019 y 2021: ¿qué área de conocimiento tuvo la mayor caída
porcentual en el número de matriculados y cuál tuvo el mayor aumento?

In [ ]:
estudiantes_2019_2021 = estudiantes_limpio[estudiantes_limpio['ano'].isin([2019, 2021])]

# Agrupar por id_area y año
area_año = estudiantes_2019_2021.groupby(['id_area', 'ano'])['total_matriculados'].sum().unstack()

# Calcular variación porcentual entre 2019 y 2021
area_año['var_pct'] = ((area_año[2021] - area_año[2019]) / area_año[2019]) * 100

# Resultados
print("Mayor caída porcentual (id_area):")
print(area_año['var_pct'].idxmin(), f"({area_año['var_pct'].min():.2f}%)")

print("\nMayor aumento porcentual (id_area):")
print(area_año['var_pct'].idxmax(), f"({area_año['var_pct'].max():.2f}%)")

El área de conocimiento tuvo la mayor caída porcentual en el número de matriculados fue el 2 que cayo un 3.44%, y el que aumento mas fue el 4 con un 6.14%.

__(i).__ ¿Cuál fue la metodología (presencial, a distancia, virtual, etc.) con mayor crecimiento porcentual en el número de matriculados entre 2015 y 2021?

In [ ]:
estudiantes_2015_2021 = estudiantes_limpio[estudiantes_limpio['ano'].isin([2015, 2021])]

# Agrupar por id_metodologia y año
met_año = estudiantes_2015_2021.groupby(['id_metodologia', 'ano'])['total_matriculados'].sum().unstack()

# Calcular variación porcentual entre 2015 y 2021
met_año['var_pct'] = ((met_año[2021] - met_año[2015]) / met_año[2015]) * 100

# Resultado
print("Metodología con mayor crecimiento porcentual:")
print(met_año['var_pct'].idxmax(), f"({met_año['var_pct'].max():.2f}%)")

La metadología con mayor crecimiento percentual teniendo en cuenta  el número de matriculados entre 2015 y 2021 fue la 3 con un 479.43% esta sospechamos que es virtual.



__(j).__ En 2018, ¿cuáles son los cinco departamentos de oferta con más matriculados y qué porcentaje del total nacional representan en conjunto?

In [ ]:

estudiantes_2018 = estudiantes_limpio[estudiantes_limpio['ano'] == 2018]

# Total nacional de matriculados en 2018
total_nacional_2018 = estudiantes_2018['total_matriculados'].sum()

# Agrupar por departamento de oferta y obtener los 5 mayores
top5_departamentos = estudiantes_2018.groupby('departamento_de_oferta_del_programa')['total_matriculados'].sum().nlargest(5)

# Porcentaje que representan los 5 departamentos en conjunto
pct_conjunto = (top5_departamentos.sum() / total_nacional_2018) * 100

print(f"Top 5 departamentos de oferta en 2018: {top5_departamentos}")

print(f"\nPorcentaje del total nacional que representan en conjunto: {pct_conjunto:.2f}%")

In [ ]:
dep_todos = estudiantes_2018.groupby('departamento_de_oferta_del_programa')['total_matriculados'].sum().sort_values(ascending=True)

# Graficar barras horizontales
plt.figure(figsize=(10, 12))
dep_todos.plot(kind='barh', color='steelblue')

plt.title('Total de Matriculados por Departamento de Oferta (2018)')
plt.xlabel('Total Matriculados')
plt.ylabel('Departamento')
plt.tight_layout()
plt.show()

Los 5 departamentos de oferta con más matriculados son: Bogotá DC, Antioquia, Valle del cauca, Atlantico y Santander. Entre esos departamentos representa en conjunto el 65.15% del total nacional.

__(k).__ ¿En cuántos registros el departamento de domicilio de la IES es distinto al departamento de oferta del programa y cuál es la IES con más matriculados en programas ofertados fuera de su departamento de domicilio?



In [ ]:
estudiantes_distinto = estudiantes_limpio[estudiantes_limpio['departamento_de_domicilio_de_la_ies'] != estudiantes_limpio['departamento_de_oferta_del_programa']]

# 1. Cantidad de registros
num_registros_distintos = len(estudiantes_distinto)

# 2. IES con más matriculados fuera de su departamento de domicilio
ies_mas_matriculados = estudiantes_distinto.groupby('institucion_de_educacion_superior__ies')['total_matriculados'].sum()

print(f"Número de registros con departamento distinto: {num_registros_distintos}")
print("\nIES con más matriculados fuera de su departamento de domicilio:")
print(f"{ies_mas_matriculados.idxmax()} ({ies_mas_matriculados.max():,} matriculados)")

El total de registros teniendo en cuenta que el departamento de domicilio de la IES es distinto al departamento de oferta del programa es de 91109 resgistros, la IES con más matriculados en programas ofertados fuera de su departamento de domicilio es: SERVICIO NACIONAL DE APRENDIZAJE SENA con 2108971 matriculados.

__(l).__ ¿Cuál es el departamento con mayor crecimiento porcentual en el número de matriculados entre 2015 y 2021?

In [ ]:


# Agrupar por departamento de oferta y año
dep_ano = estudiantes_2015_2021.groupby(['departamento_de_oferta_del_programa', 'ano'])['total_matriculados'].sum().unstack()

# Calcular variación porcentual entre 2015 y 2021
dep_ano['var_pct'] = ((dep_ano[2021] - dep_ano[2015]) / dep_ano[2015]) * 100

# Departamento con mayor crecimiento
print("Departamento con mayor crecimiento porcentual (2015-2021):")
print(f"{dep_ano['var_pct'].idxmax()} ({dep_ano['var_pct'].max():.2f}%)")

El departamento con mayor crecimiento porcentual en el número de matriculados entre 2015 y 2021 es Chocó con un aumento del 27.79%

__(m).__ ¿Cuál es el programa académico (agrupado por nombre) con mayor
crecimiento absoluto en el número de matriculados entre 2015 y 2021?

In [ ]:
# Filtrar años 2015 y 2021
estudiantes_2015_2021 = estudiantes_limpio[estudiantes_limpio['ano'].isin([2015, 2021])]

# Agrupar por programa académico y año (rellenando con 0 si no hubo registros en un año)
programas_por_año = estudiantes_2015_2021.groupby(['programa_academico', 'ano'])['total_matriculados'].sum().unstack(fill_value=0)

# Calcular crecimiento absoluto (Matriculados 2021 - Matriculados 2015)
programas_por_año['crecimiento_absoluto'] = programas_por_año[2021] - programas_por_año[2015]

# Programa académico con mayor crecimiento absoluto
prog_max = programas_por_año['crecimiento_absoluto'].idxmax()
valor_max = programas_por_año['crecimiento_absoluto'].max()

print("Programa académico con mayor crecimiento absoluto (2015-2021):")
print(f"{prog_max} (+{valor_max:,.0f} matriculados)")

Entre los años 2015 y 2021 el programa academico con mayor crecimiento absoluto es TECNOLOGÍA EN ANÁLISIS Y DESARROLLO DE SISTEMAS DE INFORMACIÓN con mas de 29397 matriculados.

__(n).__ ¿Cuál es la proporción de matriculados en posgrado frente a pregrado en cada año y cuál nivel de formación de posgrado (especialización, maestría o doctorado) creció más entre 2015 y 2021?

In [ ]:
# 1. Proporción Posgrado / Pregrado por año (id_nivel: 1=Pregrado, 2=Posgrado)
nivel_año = estudiantes_limpio.groupby(['ano', 'id_nivel'])['total_matriculados'].sum().unstack()
proporcion = nivel_año[2] / nivel_año[1]

print("Proporción (Posgrado / Pregrado) por año:")
print(proporcion.round(3))

# 2. Nivel de formación de posgrado con mayor crecimiento (2015 - 2021)
estudiantes_pos = estudiantes_limpio[(estudiantes_limpio['id_nivel'] == 2) & (estudiantes_limpio['ano'].isin([2015, 2021]))]
pos_formacion = estudiantes_pos.groupby(['id_nivel_formacion', 'ano'])['total_matriculados'].sum().unstack(fill_value=0)

crecimiento_pct = ((pos_formacion[2021] - pos_formacion[2015]) / pos_formacion[2015]) * 100

print("\nNivel de posgrado con mayor crecimiento porcentual (2015-2021):")
print(f"id_nivel_formacion {crecimiento_pct.idxmax()}: {crecimiento_pct.max():.2f}%")

Como se puede ver en la tabla se muestran las proporciones de matriculados en posgrado en cada año y el nivel de formacion de posgrado con mayor crecimieto es el 7.

__(o).__ En 2020, ¿qué tipo de institución según su carácter académico tuvo el mayor promedio de matriculados por programa?

In [ ]:
estudiantes_2020 = estudiantes_limpio[estudiantes_limpio['ano'] == 2020]

# Agrupar por id_caracter: total de matriculados y cantidad de programas únicos
promedio_pro_2020 = estudiantes_2020.groupby('id_caracter').agg(
    total_matriculados=('total_matriculados', 'sum'),
    total_programas=('codigo_snies_delprograma', 'nunique')
)

# Calcular el promedio de matriculados por programa
promedio_pro_2020['promedio_por_programa'] = promedio_pro_2020['total_matriculados'] / promedio_pro_2020['total_programas']

# Resultado
print("Promedio de matriculados por programa según carácter académico (2020):")
print(promedio_pro_2020['promedio_por_programa'].round(2))

print("\nCarácter académico con mayor promedio:")
print(f"id_caracter {promedio_pro_2020['promedio_por_programa'].idxmax()}: {promedio_pro_2020['promedio_por_programa'].max():.2f} matriculados/programa")

En el año 2020 la intitucion segun su carácter académico que tuvo el mayor promedio de matriculados por programa es la 2 con 362.68 matriculados por programa.

__(p).__ En 2021, ¿qué porcentaje del total de matriculados se encuentra en
Bogotá D.C., Medellín, Cali y Barranquilla en conjunto (municipio de oferta
del programa)?

In [ ]:
# 1. Filtrar año 2021
estudiantes_2021 = estudiantes_limpio[estudiantes_limpio["ano"] == 2021]

# 2. Definir ciudades de oferta a consultar
ciudades = ["BOGOTA D.C.", "MEDELLIN", "CALI", "BARRANQUILLA"]

# 3. Totales y cálculo de porcentaje
total_nacional = estudiantes_2021["total_matriculados"].sum()
total_ciudades = estudiantes_2021[
    estudiantes_2021["municipio_de_oferta_del_programa"].isin(ciudades)
]["total_matriculados"].sum()

porcentaje = (total_ciudades / total_nacional) * 100

print(f"Matriculados en Bogotá, Medellín, Cali y Barranquilla (2021): {total_ciudades:,}")
print(f"{porcentaje:.2f}%")

Para el año 2021, el porcentaje del total de matriculados que se concentra en las cuatro principales ciudades (Bogotá D.C., Medellín, Cali y Barranquilla) en conjunto es del:  20.45%

__(q).__ ¿Hay diferencias entre el número de matriculados del primer y del segundo semestre en cada año? ¿En qué año fue mayor la diferencia?

In [ ]:
# 1. Agrupar por año y semestre
semestres = (
    estudiantes_limpio.groupby(["ano", "semestre"])["total_matriculados"]
    .sum()
    .unstack()
)

# 2. Calcular la diferencia absoluta entre semestres
semestres["diferencia"] = (semestres[1] - semestres[2]).abs()

# 3. Año con la mayor diferencia
ano_max_diff = semestres["diferencia"].idxmax()
max_diff = semestres["diferencia"].max()

print("Matriculados por año y semestre:\n", semestres[[1, 2, "diferencia"]])
print(f"\nEl año con mayor diferencia entre semestres fue {ano_max_diff} ({max_diff:,} matriculados)")

El año con mayor diferencia entre semestres fue 2021 con 1567867 matriculados

__(r).__ En Medicina, ¿cuál es el departamento con más matriculados en 2021 y
cómo ha sido su evolución desde 2015?

In [ ]:


# 1. Filtrar programa de Medicina en 2021
medicina = estudiantes_limpio[
    estudiantes_limpio["programa_academico"] == "MEDICINA"
]
medicina_2021 = (
    medicina[medicina["ano"] == 2021]
    .groupby("departamento_de_oferta_del_programa")["total_matriculados"]
    .sum()
)

dpto_lider = medicina_2021.idxmax()
num_lider = medicina_2021.max()

print(
    f"Resultado: {dpto_lider} ({num_lider:,} )"
)

# 2. Evolución del departamento líder de 2015 a 2021
evolucion_dpto = (
    medicina[
        (medicina["departamento_de_oferta_del_programa"] == dpto_lider)
        & (medicina["ano"].between(2015, 2021))
    ]
    .groupby("ano")["total_matriculados"]
    .sum()
)

# 3. Gráfica de evolución
evolucion_dpto.plot(
    kind="line",
    marker="o",
    figsize=(8, 4),
    title=f"Evolución Medicina en {dpto_lider} (2015-2021)",
)
plt.ylabel("Total Matriculados")
plt.grid(True)
plt.show()

El Departamento con más matriculados en Medicina en el 2021 es: BOGOTA DC (CAPITAL) con 13140 matriculados

__(s).__ En 2019, ¿cuál es la correlación entre el número de programas ofertados y
el total de matriculados por departamento de oferta? Acompañe la respuesta
con un diagrama de dispersión.


In [ ]:
import matplotlib.pyplot as plt

# 1. Agrupar datos de 2019
dpto_2019 = (
    estudiantes_limpio[estudiantes_limpio["ano"] == 2019]
    .groupby("departamento_de_oferta_del_programa")
    .agg(
        num_programas=("codigo_snies_delprograma", "nunique"),
        total_matriculados=("total_matriculados", "sum"),
    )
)

# 2. Correlación
print(
    f"Correlación : {dpto_2019['num_programas'].corr(dpto_2019['total_matriculados']):.4f}"
)

# 3. Diagrama de dispersión alternativo (Matplotlib puro con puntos flotantes)
plt.scatter(
    dpto_2019["num_programas"],
    dpto_2019["total_matriculados"],
    color="darkcyan",
    alpha=0.7,
    edgecolors="black",
)
plt.title("Relación Programas Ofertados vs. Matriculados (2019)")
plt.xlabel("Programas Ofertados")
plt.ylabel("Total Matriculados")
plt.show()

la correlación entre el número de programas ofertados y el total de matriculados por departamento de oferta es de 0.9894

__(t).__ Seleccione una IES que sea de su interés y analice la evolución de su matrícula entre 2015 y 2021 según género, nivel de formación y área de conocimiento. ¿Qué conclusiones puede sacar?

In [ ]:
# Definir 'ies' filtrando la institución y los años de interés
ies = estudiantes_limpio[
    (
        estudiantes_limpio["institucion_de_educacion_superior__ies"]
        == "UNIVERSIDAD NACIONAL DE COLOMBIA"
    )
    & (estudiantes_limpio["ano"].between(2015, 2021))
]

# 1. Tabla de números por Género
tabla_genero = ies.pivot_table(
    index="ano",
    columns="id_genero",
    values="total_matriculados",
    aggfunc="sum",
)

print(tabla_genero)

# 2. Tabla de números por Área de Conocimiento
tabla_area = ies.pivot_table(
    index="ano",
    columns="id_area",
    values="total_matriculados",
    aggfunc="sum",
)

print(tabla_area)

La IES que se elgio fue UNIVERSIDAD NACIONAL DE COLOMBIA.

Entre 2015 y 2021, la matrícula creció de 52631 a 58351 estudiantes, superando la caída del 18.7% en 2020 por la pandemia. La participación femenina subió al 38.7%. Sobresale la concentración en Ingeniería y Arquitectura (Área 8) con el 46% del total, el crecimiento en Ciencias (Área 5) y Agronomía (Área 9), y la caída superior al 50% en Artes (Área 3).